In [1]:
import pandas as pd
from pathlib import Path


data_path = Path('../data')
input_path = Path('input')
orders_file = data_path / 'olist_orders_dataset.csv'

if orders_file.exists():
    df = pd.read_csv(orders_file)
else:
    df = pd.read_csv(input_path / 'olist_orders_2018_01.csv')
    print('Using January export for the introductory checks')

Using January export for the introductory checks


In [2]:
print(data_path)
print(orders_file)
print(orders_file.exists())

../data
../data/olist_orders_dataset.csv
False


In [3]:
csv_files = list(data_path.glob("*.csv"))
csv_files

[]

In [4]:
# Jan–Mar only
all_monthly_files = sorted(input_path.glob('*.csv'))
monthly_files = []
for file in all_monthly_files:
    dates = pd.to_datetime(pd.read_csv(file, usecols=['order_purchase_timestamp'])['order_purchase_timestamp'], errors='coerce')
    months = set(dates.dt.to_period('M').astype(str))
    if dates.empty or dates.isna().any() or len(months) != 1:
        raise ValueError(f'Invalid dates or mixed months in {file.name}')
    if next(iter(months)) in {'2018-01', '2018-02', '2018-03'}:
        monthly_files.append(file)
if len(monthly_files) != 3:
    raise ValueError(f'Expected January–March exports, found {[f.name for f in monthly_files]}')
monthly_files

[PosixPath('input/olist_orders_2018_01.csv'),
 PosixPath('input/olist_orders_2018_02.csv'),
 PosixPath('input/olist_orders_2018_03.csv')]

In [5]:
dataframes = []

for file in monthly_files:
    temp_df = pd.read_csv(file)
    temp_df["source_file"] = file.name
    dataframes.append(temp_df)

In [6]:
combined_df = pd.concat(dataframes, ignore_index=True)

print(combined_df.shape)

(21208, 10)


In [7]:
sum(len(pd.read_csv(file)) for file in monthly_files)

21208

In [8]:
len(combined_df)

21208

In [9]:
for file in monthly_files:
    print(file.name)

olist_orders_2018_01.csv
olist_orders_2018_02.csv
olist_orders_2018_03.csv


In [10]:
for file in csv_files:
    print(file.name)

In [11]:
[file.name for file in csv_files]

[]

In [12]:
for file in csv_files:
    print(file.name)
    print(file.stem)
    print(file.suffix)
    print()

In [13]:
print(df.shape)
print(df.columns.tolist())

(7269, 9)
['customer_state', 'order_id', 'order_status', 'order_purchase_timestamp', 'has_items', 'item_count', 'total_price', 'freight_total', 'order_total']


In [14]:
def validate_dataframe(df, required_columns, key_column=None):
    rows, columns = df.shape

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    missing_by_column = {
        column: int(df[column].isna().sum())
        for column in df.columns
    }

    duplicate_key_count = None

    if key_column is not None and key_column in df.columns:
        duplicate_key_count = int(
            df.duplicated(subset=[key_column]).sum()
        )

    return {
        "rows": int(rows),
        "columns": int(columns),
        "duplicate_key_count": duplicate_key_count,
        "missing_values": int(df.isna().sum().sum()),
        "column_names": list(df.columns),
        "missing_required_columns": missing_columns,
        "missing_by_column": missing_by_column
    }

In [15]:
required_columns = ["order_id", "order_status", "order_purchase_timestamp"]

In [16]:
validate_dataframe(
    df,
    required_columns,
    key_column="order_id"
)

{'rows': 7269,
 'columns': 9,
 'duplicate_key_count': 0,
 'missing_values': 147,
 'column_names': ['customer_state',
  'order_id',
  'order_status',
  'order_purchase_timestamp',
  'has_items',
  'item_count',
  'total_price',
  'freight_total',
  'order_total'],
 'missing_required_columns': [],
 'missing_by_column': {'customer_state': 0,
  'order_id': 0,
  'order_status': 0,
  'order_purchase_timestamp': 0,
  'has_items': 0,
  'item_count': 0,
  'total_price': 49,
  'freight_total': 49,
  'order_total': 49}}

In [17]:
required_columns_test = ["order_id", "order_status", "order_purchase_timestamp", "fake_column"]

In [18]:
validate_dataframe(
    df,
    required_columns_test,
    key_column="order_id"
)

{'rows': 7269,
 'columns': 9,
 'duplicate_key_count': 0,
 'missing_values': 147,
 'column_names': ['customer_state',
  'order_id',
  'order_status',
  'order_purchase_timestamp',
  'has_items',
  'item_count',
  'total_price',
  'freight_total',
  'order_total'],
 'missing_required_columns': ['fake_column'],
 'missing_by_column': {'customer_state': 0,
  'order_id': 0,
  'order_status': 0,
  'order_purchase_timestamp': 0,
  'has_items': 0,
  'item_count': 0,
  'total_price': 49,
  'freight_total': 49,
  'order_total': 49}}

In [19]:
validation_report = validate_dataframe(
    combined_df,
    [
        'order_id', 'customer_state', 'order_status',
        'order_purchase_timestamp', 'has_items', 'item_count',
        'total_price', 'freight_total', 'order_total', 'source_file'
    ],
    key_column='order_id'
)

if validation_report['missing_required_columns']:
    raise ValueError(f"Missing columns: {validation_report['missing_required_columns']}")
if validation_report['duplicate_key_count']:
    raise ValueError(f"Duplicate order IDs: {validation_report['duplicate_key_count']}")

# Check files before export
expected_months = {'2018-01', '2018-02', '2018-03'}
months_by_file = {}
for file in monthly_files:
    file_dates = pd.to_datetime(
        combined_df.loc[combined_df['source_file'] == file.name, 'order_purchase_timestamp'],
        errors='coerce'
    )
    if file_dates.empty or file_dates.isna().any():
        raise ValueError(f'Missing or invalid dates in {file.name}')
    file_months = set(file_dates.dt.to_period('M').astype(str))
    if len(file_months) != 1:
        raise ValueError(f'Expected one month per file: {file.name} has {sorted(file_months)}')
    months_by_file[file.name] = next(iter(file_months))
if set(months_by_file.values()) != expected_months:
    raise ValueError(f'Expected Jan–Mar 2018, found {months_by_file}')

print('Months:', months_by_file)
validation_report

Months: {'olist_orders_2018_01.csv': '2018-01', 'olist_orders_2018_02.csv': '2018-02', 'olist_orders_2018_03.csv': '2018-03'}


{'rows': 21208,
 'columns': 10,
 'duplicate_key_count': 0,
 'missing_values': 318,
 'column_names': ['customer_state',
  'order_id',
  'order_status',
  'order_purchase_timestamp',
  'has_items',
  'item_count',
  'total_price',
  'freight_total',
  'order_total',
  'source_file'],
 'missing_required_columns': [],
 'missing_by_column': {'customer_state': 0,
  'order_id': 0,
  'order_status': 0,
  'order_purchase_timestamp': 0,
  'has_items': 0,
  'item_count': 0,
  'total_price': 106,
  'freight_total': 106,
  'order_total': 106,
  'source_file': 0}}

In [20]:
import importlib.util
if importlib.util.find_spec('openpyxl') is None:
    raise ImportError('Install openpyxl once with: pip install openpyxl')

In [21]:
import json

output_path = Path("output")
output_path.mkdir(exist_ok=True)

combined_df.to_csv(
    output_path / "orders_jan_mar_2018.csv",
    index=False
)

combined_df.to_excel(
    output_path / "orders_jan_mar_2018.xlsx",
    index=False
)

with open(
    output_path / "validation_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        validation_report,
        file,
        indent=4,
        ensure_ascii=False
    )

In [22]:
for file in output_path.iterdir():
    print(file.name, file.stat().st_size)

orders_jan_mar_2018.csv 2475743
validation_report.json 716
orders_jan_mar_2018.xlsx 1518110


In [23]:
def run_pipeline(input_path, output_path, required_columns, key_column, expected_months=None):
    input_path = Path(input_path)
    output_path = Path(output_path)
    csv_files = sorted(input_path.glob('*.csv'))

    if not csv_files:
        raise FileNotFoundError(f'No CSV files found in {input_path.resolve()}')

    dataframes = []
    months_by_file = {}
    selected_months = set(expected_months) if expected_months is not None else None
    for file in csv_files:
        temp_df = pd.read_csv(file)
        if 'order_purchase_timestamp' not in temp_df.columns:
            raise ValueError(f'Missing order_purchase_timestamp in {file.name}')
        dates = pd.to_datetime(temp_df['order_purchase_timestamp'], errors='coerce')
        if dates.empty or dates.isna().any():
            raise ValueError(f'Invalid or missing dates in {file.name}')
        file_months = set(dates.dt.to_period('M').astype(str))
        if len(file_months) != 1:
            raise ValueError(f'Expected one month per file: {file.name} has {sorted(file_months)}')
        month = next(iter(file_months))
        if selected_months is not None and month not in selected_months:
            continue
        if month in months_by_file.values():
            raise ValueError(f'Multiple CSV exports for {month}')
        months_by_file[file.name] = month
        temp_df['source_file'] = file.name
        dataframes.append(temp_df)

    if selected_months is not None and set(months_by_file.values()) != selected_months:
        raise ValueError(f'Expected {sorted(selected_months)}, found {months_by_file}')
    if not dataframes:
        raise ValueError('No matching monthly CSV files found')

    combined_df = pd.concat(dataframes, ignore_index=True)
    report = validate_dataframe(combined_df, required_columns, key_column)
    if report['missing_required_columns']:
        raise ValueError(f"Missing columns: {report['missing_required_columns']}")
    if report['duplicate_key_count']:
        raise ValueError(f"Duplicate {key_column} values: {report['duplicate_key_count']}")

    output_path.mkdir(parents=True, exist_ok=True)
    combined_df.to_csv(output_path / 'combined_orders.csv', index=False)
    combined_df.to_excel(output_path / 'combined_orders.xlsx', index=False)
    with (output_path / 'validation_report.json').open('w', encoding='utf-8') as f:
        json.dump(report, f, indent=4, ensure_ascii=False)
    return combined_df, report

In [24]:
pipeline_required_columns = [
    "order_id",
    "customer_state",
    "order_status",
    "order_purchase_timestamp",
    "has_items",
    "item_count",
    "total_price",
    "freight_total",
    "order_total",
    "source_file"
]

In [25]:
pipeline_df, pipeline_report = run_pipeline(
    input_path='input',
    output_path='pipeline_output',
    required_columns=pipeline_required_columns,
    key_column='order_id',
    expected_months={'2018-01', '2018-02', '2018-03'}
)

pipeline_report

{'rows': 21208,
 'columns': 10,
 'duplicate_key_count': 0,
 'missing_values': 318,
 'column_names': ['customer_state',
  'order_id',
  'order_status',
  'order_purchase_timestamp',
  'has_items',
  'item_count',
  'total_price',
  'freight_total',
  'order_total',
  'source_file'],
 'missing_required_columns': [],
 'missing_by_column': {'customer_state': 0,
  'order_id': 0,
  'order_status': 0,
  'order_purchase_timestamp': 0,
  'has_items': 0,
  'item_count': 0,
  'total_price': 106,
  'freight_total': 106,
  'order_total': 106,
  'source_file': 0}}

In [26]:
april_files = [f.name for f in all_monthly_files if f not in monthly_files]
print('Other monthly exports:', april_files)

Other monthly exports: ['olist_orders_2018_04.csv']


In [27]:
for file in Path('output').glob('*'):
    print(file.name, file.stat().st_size)

orders_jan_mar_2018.csv 2475743
validation_report.json 716
orders_jan_mar_2018.xlsx 1518110


In [28]:
pipeline_report

{'rows': 21208,
 'columns': 10,
 'duplicate_key_count': 0,
 'missing_values': 318,
 'column_names': ['customer_state',
  'order_id',
  'order_status',
  'order_purchase_timestamp',
  'has_items',
  'item_count',
  'total_price',
  'freight_total',
  'order_total',
  'source_file'],
 'missing_required_columns': [],
 'missing_by_column': {'customer_state': 0,
  'order_id': 0,
  'order_status': 0,
  'order_purchase_timestamp': 0,
  'has_items': 0,
  'item_count': 0,
  'total_price': 106,
  'freight_total': 106,
  'order_total': 106,
  'source_file': 0}}

In [29]:
print(len(pipeline_df))
print(pipeline_df["source_file"].value_counts())

21208
source_file
olist_orders_2018_01.csv    7269
olist_orders_2018_03.csv    7211
olist_orders_2018_02.csv    6728
Name: count, dtype: int64


In [30]:
from pathlib import Path

[file.name for file in Path("input").glob("*.csv")]

['olist_orders_2018_01.csv',
 'olist_orders_2018_02.csv',
 'olist_orders_2018_04.csv',
 'olist_orders_2018_03.csv']

In [31]:
print(len(pipeline_df))
print(pipeline_df["source_file"].value_counts())

21208
source_file
olist_orders_2018_01.csv    7269
olist_orders_2018_03.csv    7211
olist_orders_2018_02.csv    6728
Name: count, dtype: int64


In [32]:
print('Rows:', len(pipeline_df))
print('Unique orders:', pipeline_df['order_id'].nunique())
print('Months:', sorted(pd.to_datetime(pipeline_df['order_purchase_timestamp']).dt.to_period('M').astype(str).unique()))
print('Source files:', pipeline_df['source_file'].nunique())

Rows: 21208
Unique orders: 21208
Months: ['2018-01', '2018-02', '2018-03']
Source files: 3


## April Gate

Use the April CSV from `input/`

In [33]:
# April Gate
april_df, april_report = run_pipeline(
    input_path='input',
    output_path='april_output',
    required_columns=pipeline_required_columns,
    key_column='order_id',
    expected_months={'2018-04'}
)

print('April rows:', len(april_df))
print('Unique orders:', april_df['order_id'].nunique())
print('Orders without items:', int((april_df['has_items'] == 0).sum()))
print('Goods total:', round(april_df['total_price'].sum(), 2))
print('Freight total:', round(april_df['freight_total'].sum(), 2))
print('Order total:', round(april_df['order_total'].sum(), 2))

April rows: 6939
Unique orders: 6939
Orders without items: 5
Goods total: 996647.75
Freight total: 163050.29
Order total: 1159698.04


In [34]:
# Check April totals
import numpy as np

for column in ['has_items', 'item_count', 'total_price', 'freight_total', 'order_total']:
    april_df[column] = pd.to_numeric(april_df[column], errors='raise')

if not april_df['has_items'].isin([0, 1]).all():
    raise ValueError('has_items must be 0 or 1')
if (april_df['item_count'] < 0).any():
    raise ValueError('Negative item_count found')
if not ((april_df['has_items'] == 0) == (april_df['item_count'] == 0)).all():
    raise ValueError('has_items does not match item_count')

has_items = april_df['has_items'] == 1
if april_df.loc[has_items, ['total_price', 'freight_total', 'order_total']].isna().any().any():
    raise ValueError('Missing totals on orders with items')
if not np.allclose(
    april_df.loc[has_items, 'order_total'],
    april_df.loc[has_items, 'total_price'] + april_df.loc[has_items, 'freight_total'],
    atol=0.01, rtol=0
):
    raise ValueError('order_total does not match price + freight')

april_controls = {
    'row_count': len(april_df),
    'unique_orders': april_df['order_id'].nunique(),
    'goods_sum': round(april_df['total_price'].sum(), 2),
    'freight_sum': round(april_df['freight_total'].sum(), 2),
    'order_sum': round(april_df['order_total'].sum(), 2),
    'orders_without_items': int((april_df['has_items'] == 0).sum())
}
print(april_controls)

# April controls from the supplied SQL monthly export
sql_april_controls = {'row_count': 6939, 'unique_orders': 6939, 'goods_sum': 996647.75, 'freight_sum': 163050.29, 'order_sum': 1159698.04, 'orders_without_items': 5}
if sql_april_controls is not None:
    for key, actual in april_controls.items():
        expected = sql_april_controls[key]
        if not np.isclose(actual, expected, atol=0.01, rtol=0):
            raise AssertionError(f'{key}: Pandas={actual}, SQL={expected}')
    print('April pipeline matches the supplied April export')
else:
    print('April Pandas checks passed; SQL comparison not yet entered')

{'row_count': 6939, 'unique_orders': 6939, 'goods_sum': np.float64(996647.75), 'freight_sum': np.float64(163050.29), 'order_sum': np.float64(1159698.04), 'orders_without_items': 5}
April pipeline matches the supplied April export


## Notes

January–March and April are exported separately

April totals are checked against the source CSV; SQL comparison needs its own control results